In [109]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import io
import requests
import matplotlib.pyplot as plt
# seaborn is a better tool for this with more examples 
import seaborn as sns


df_melanoma = pd.read_csv('melanoma_mitoses.csv', sep=';')

In [110]:
df_melanoma.head()

,ID_prog,age,sex,year,histology,site,pT,pN,pM,TNM_stage,breslow,clark,ulcerazione,regressione,til,growth,positive_slnb,mitoses,status,survival
0,1,31,F,2015,Nodular melanoma,Trunk,pT4,N3c,M0,III,>=4,IV,Present,Absent,Absent,Vertical,0,3,0,648
1,2,53,M,2017,Superficial spreading melanoma,Trunk,pT1,N0,M0,I,< 0.76,II,Absent,Present,Present,Vertical,0,0,0,1147
2,3,67,F,2015,Superficial spreading melanoma,Upper limb,pT1,N0,M0,I,0.76-1.50,IV,Absent,Absent,Present,Vertical,0,0,0,1948
3,4,29,F,2015,Superficial spreading melanoma,Trunk,pT1,N1a,M0,III,< 0.76,III,Absent,Absent,Present,Vertical,1,0,0,2170
4,5,57,F,2015,Superficial spreading melanoma,Lower limb,pT1,N0,M0,I,< 0.76,II,Absent,Absent,Present,Radial,0,0,0,1960


In [111]:
df_melanoma.shape

(2255, 20)

In [112]:
df_melanoma.columns

Index(['ID_prog', 'age', 'sex', 'year', 'histology', 'site', 'pT', 'pN', 'pM',
       'TNM_stage', 'breslow', 'clark', 'ulcerazione', 'regressione', 'til',
       'growth', 'positive_slnb', 'mitoses', 'status', 'survival'],
      dtype='object')

In [113]:
df_melanoma[['survival', 'status']]

,survival,status
0,648,0
1,1147,0
2,1948,0
3,2170,0
4,1960,0
...,...,...
2250,1275,0
2251,436,1
2252,1297,0
2253,18,1


## Missing values analysis
Missing Completely at Random (MCAR) — the missingness has no relationship to anything. Rare in practice.
Missing at Random (MAR) — the missingness depends on other observed variables. For example, Clark level might be missing more often for thicker tumours because clinicians didn't bother recording it when Breslow was clearly the more relevant measure.
Missing Not at Random (MNAR) — the missingness depends on the missing value itself. For example, if regressione is missing because pathologists only record it when they see it, then the nulls might actually be informative — they could effectively mean "not observed/not present."


features with more than 15–20% missing values were excluded because imputation of patient-specific histological findings would introduce misleading information


In [114]:
# Now start to clean the data
# In order to make this model work we will need to encode
# many of the variables to make them numeric

# First we can drop unnecessary columns

df_melanoma_clean = df_melanoma.drop(columns=[ 'survival',   # leakage: another way of showing status the variable we are trying to predict
                                           'ID_prog',    # row identifier, no predictive value
                                           'clark',      # this is an alternative measure of tumour depth similiar to breslow. Data has many nulls so will use breslow instead
                                           'regressione', # there are over 500 nulls in 1804 rows in the testing data. Regression in patients is independent so imputation isn't valid
                                           'growth',      # same argument as for regression
                                           'year'])      # not needed

# ORDINAL ENCODINGS 
# This for the variables where we care about the order
# so e.g. the larger the breslow_map (thicker the tumor) 
# the more dangerous it is, so is encoded as 5

tnm_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4}
df_melanoma_clean['TNM_stage'] = df_melanoma_clean['TNM_stage'].map(tnm_map)

pt_map = {'pT1': 1, 'pT2': 2, 'pT3': 3, 'pT4': 4}
df_melanoma_clean['pT'] = df_melanoma_clean['pT'].map(pt_map)

breslow_map = {'< 0.76': 1, '0.76-1.50': 2, '1.51-3.99': 3, '>=4': 4}
df_melanoma_clean['breslow'] = df_melanoma_clean['breslow'].map(breslow_map)

# clark_map = {'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5}
# df_melanoma_clean['clark'] = df_melanoma_clean['clark'].map(clark_map)

# BINARY ENCODINGS
# Simple categorization between 2 options

df_melanoma_clean['sex']         = df_melanoma_clean['sex'].map({'F': 0, 'M': 1})
df_melanoma_clean['ulcerazione'] = df_melanoma_clean['ulcerazione'].map({'Absent': 0, 'Present': 1})
df_melanoma_clean['til']         = df_melanoma_clean['til'].map({'Absent': 0, 'Present': 1})
# df_melanoma_clean['growth']      = df_melanoma_clean['growth'].map({'Radial': 0, 'Vertical': 1})
# df_melanoma_clean['regressione'] = df_melanoma_clean['regressione'].map({'Absent': 0, 'Present': 1})
df_melanoma_clean['pM']          = df_melanoma_clean['pM'].map({'M0': 0, 'M1': 1})
df_melanoma_clean['positive_slnb'] = df_melanoma_clean['positive_slnb'].map({'0': 0, '1': 1, '>1': 2})

# ORDINAL pN 
pn_map = {'N0': 0, 'N1a': 1, 'N1b': 1, 'N1c': 1,
           'N2a': 2, 'N2b': 2, 'N2c': 2,
           'N3':  3, 'N3c': 3}
df_melanoma_clean['pN'] = df_melanoma_clean['pN'].map(pn_map)

# ONE-HOT ENCODE 
# Columns we need to encode but don't care about the order
# This is a standard method of working with this kind of data
# drop_first=True drops one dummy per group to avoid multicollinearity
df_melanoma_clean = pd.get_dummies(df_melanoma_clean, columns=['histology', 'site'], drop_first=True)

# LOG-TRANSFORM 
# This is the recommended way where you have a wide range of values 
# mitosis runs from 0 to 55 and is very skewed to the lower end of 
# that range
df_melanoma_clean['mitoses_log'] = np.log1p(df_melanoma_clean['mitoses'])

# ... and drop the original mitosis column
df_melanoma_clean = df_melanoma_clean.drop(columns=['mitoses'])

In [115]:
df_melanoma_clean.columns

Index(['age', 'sex', 'pT', 'pN', 'pM', 'TNM_stage', 'breslow', 'ulcerazione',
       'til', 'positive_slnb', 'status', 'histology_Desmoplastic melanoma',
       'histology_Lentigo maligna', 'histology_Malignant melanoma',
       'histology_Nodular melanoma', 'histology_Spitzoid melanoma',
       'histology_Superficial spreading melanoma', 'site_Head',
       'site_Lower limb', 'site_Trunk', 'site_Upper limb', 'mitoses_log'],
      dtype='object')

In [116]:
df_melanoma_clean.shape

(2255, 22)

In [117]:
#### We have to check for nulls in the values to make sure we can run the model.

In [118]:
#rows, cols = df_melanoma_clean.isnull().values.nonzero()
#for r, c in zip(rows, cols):
#    print(f"Row {r}, Column: {df_melanoma_clean.columns[c]}")
df_melanoma_clean.isnull().sum()

#import seaborn as sns
#import matplotlib.pyplot as plt

#sns.heatmap(df_melanoma_clean.isnull(), yticklabels=False, cbar=False, cmap='viridis')
#plt.title('Missing Value Map')
#plt.show()

age                                           0
sex                                           0
pT                                            3
pN                                            8
pM                                           13
TNM_stage                                    16
breslow                                       4
ulcerazione                                  25
til                                         143
positive_slnb                                 0
status                                        0
histology_Desmoplastic melanoma               0
histology_Lentigo maligna                     0
histology_Malignant melanoma                  0
histology_Nodular melanoma                    0
histology_Spitzoid melanoma                   0
histology_Superficial spreading melanoma      0
site_Head                                     0
site_Lower limb                               0
site_Trunk                                    0
site_Upper limb                         

Given imputation doesn't really work in this case as each row specifies a patient, and there is no valid reason to be able to impute between 2 adjacent rows.

Check how many rows we'd lose by dropping all nulls and whether nulls overlapCheck how many rows we'd lose by dropping all nulls and whether nulls overlapThat's a very clean result. You'd lose 189 rows, leaving you with 2,066 patients — that's only an 8% reduction. Plenty of data.

The interesting finding is that 133 of those 189 rows are cases where TIL is the only missing value. So TIL is driving most of the row loss. The other columns (pT, pN, pM, breslow, ulcerazione, site, TNM_stage) contribute relatively few additional drops because their nulls are small in number and presumably overlap with each other somewhat (the staging columns especially — if pT is missing, TNM_stage is likely missing too).

Dropping rows is the right call here. You keep 2,066 completely clean rows with no imputed values, which is more than enough for logistic regression. And your justification is straightforward: all missing values represent unrecorded patient-specific clinical observations where imputation would be clinically meaningless.


In [119]:
df_melanoma_clean.shape

(2255, 22)

In [120]:
df_melanoma_clean['status'].value_counts()

status
0    1980
1     275
Name: count, dtype: int64

In [121]:
# drop rows that contain a null value
df_melanoma_clean = df_melanoma_clean.dropna()
df_melanoma_clean.shape

(2079, 22)

In [122]:
# check we have no nulls here
df_melanoma_clean.isnull().sum()

age                                         0
sex                                         0
pT                                          0
pN                                          0
pM                                          0
TNM_stage                                   0
breslow                                     0
ulcerazione                                 0
til                                         0
positive_slnb                               0
status                                      0
histology_Desmoplastic melanoma             0
histology_Lentigo maligna                   0
histology_Malignant melanoma                0
histology_Nodular melanoma                  0
histology_Spitzoid melanoma                 0
histology_Superficial spreading melanoma    0
site_Head                                   0
site_Lower limb                             0
site_Trunk                                  0
site_Upper limb                             0
mitoses_log                       

In [123]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# Prepare data
# X here is the training data which must be a table, so 2D. Here is a good explaination..
#The 1D vs 2D distinction is purely about shape:
#  - A Series [1, 2, 3] has shape (3,) — 1D
#  - A DataFrame with one column has shape (3, 1) — 2D
#  - A DataFrame with two columns has shape (3, 2) — 2D

#  So in your case, Season is not the row identifier — it's a feature just like Operator. The row index is separate and
#  implicit. sklearn doesn't use it at all; it just sees a table of values where each row is one observation and each
#  column is one feature.
#  so we could have X = df_new[['Season', 'Operator', 'Locomotive'.....]] we are just creating a table.

X = df_melanoma_clean.drop(columns=['status'])

# only if you need to encode your categories
# ski-kitlearn can't cope with catagories such as "spring" so we have to encode them into 
# expand your 2-column X into ~27 binary columns (4 seasons + 24 operators — minus one each to avoid
# redundancy, a concept called the dummy variable trap). get_dummies handles this with drop_first=True:
# X_encoded = pd.get_dummies(X, columns=['Season', 'Operator'], drop_first=True)


# for Y that must be a series (aka a list) which is expected for target variable.
y = df_melanoma_clean['status']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

Training set size: 1663
Test set size: 416


In [124]:
# Fit logistic regression model
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


## Definitions of the Classification Report

| Classification | Definition | Comments |
| -------------- | ---------- | -------- |
|  **Precision** | Everything the model *predicted* as that class, how many were actually correct|  So for "Died": of all the patients the model flagged as deaths, 68% genuinely died. The other 32% were false alarms (survivors incorrectly labelled as deaths). |
| **Recall** | Everything that *actually is* that class, how many did the model catch? | For "Died": of the 57 real deaths in your test set, the model only found 49% of them. It missed more than half. In a clinical context this is the dangerous number — those are patients who died but the model said they'd survive. |
| **F1-score** | Harmonic mean of precision and recall. It penalises you if either one is poor, so it's a better single-number summary than accuracy when classes are imbalanced. | Your "Died" F1 of 0.57 is telling you the model is genuinely struggling with the minority class. |
| **Support** | Actual count of that class in your test set. | You have 359 survivors and 57 deaths. This immediately shows the imbalance problem — the model has seen far more "survived" examples to learn from. |

---

**The three row summaries at the bottom:**

**Accuracy** — overall, what fraction of all 416 predictions were correct. 90% sounds good, but this is the misleading number for your dataset. A model that predicted "Survived" for every single patient would score 359/416 = 86% accuracy without learning anything at all.

**Macro average** — averages precision, recall, and F1 across both classes equally, regardless of how many patients are in each class. This treats "Died" and "Survived" as equally important. Your macro recall of 0.73 is telling you the real story — averaged fairly across both classes, the model is nowhere near as good as 90% suggests.

**Weighted average** — averages the same metrics but weights each class by its support count. Because "Survived" (359 cases) dominates, the weighted average is pulled strongly toward those numbers and flatters the model. This is why it sits close to the accuracy figure.

---



## Solution
The model is behaving poorly due to the imbalance in the data between the deaths and the survivors. The standard way of fixing this is to use the imbalance value to allow the model to weigh correctly the imbalance

In [125]:
# Make predictions on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1

# Evaluate performance
print("Classification Report on Test Set:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Survived', 'Dead']))

Classification Report on Test Set:
              precision    recall  f1-score   support

    Survived       0.92      0.96      0.94       359
        Dead       0.68      0.49      0.57        57

    accuracy                           0.90       416
   macro avg       0.80      0.73      0.76       416
weighted avg       0.89      0.90      0.89       416



In [126]:
# Fit logistic regression model
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [127]:
# Make predictions on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1

# Evaluate performance
print("Classification Report on Test Set:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Survived', 'Dead']))

Classification Report on Test Set:
              precision    recall  f1-score   support

    Survived       0.97      0.77      0.86       359
        Dead       0.37      0.86      0.52        57

    accuracy                           0.78       416
   macro avg       0.67      0.81      0.69       416
weighted avg       0.89      0.78      0.81       416



This is better but still not there. We can do better

In [128]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',  # corrects for 7:1 class imbalance
        solver='lbfgs',
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

,steps,"[('scaler', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [129]:
# Make predictions on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probability of class 1

# Evaluate performance
print("Classification Report on Test Set:")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['Survived', 'Dead']))

Classification Report on Test Set:
              precision    recall  f1-score   support

    Survived       0.98      0.78      0.87       359
        Dead       0.39      0.88      0.54        57

    accuracy                           0.80       416
   macro avg       0.68      0.83      0.70       416
weighted avg       0.90      0.80      0.82       416



## To Do
- Understand the accuracy of the model
-- done this and shown with improvements, can maybe do some plots here
-- also need citations as to why we would do this.
- decide which illustrations / visuals to use and why
- look at the swtich to survival as well